# 20. Q4 Ontology Semantic Contract Audit v01.1

Direction:

```text
Ontology-based Semantic Contract Audit for Text2SQL
```

Mục tiêu notebook:

- Giữ `19_q4_ontology_semantic_contract_audit_v01.ipynb` làm audit baseline.
- Củng cố chính audit framework trước khi dùng nó làm tiêu chuẩn nghiệm thu ontology.
- Không sửa ontology artifacts trong notebook này.

Hardening scope:

- A1: Không xem `required_parameters` là bằng chứng SQL đã preserve semantic parameter.
- A2: Bổ sung kiểm tra type, range, dependency cho parameters.
- A3: Siết T04 zero-count test: guard allowed, SQL executed, fixture có zero-count, output thiếu zero-count.
- A4: Tăng chất lượng unsupported relation và method composition tests.
- A5: Bổ sung negative mutation tests cho audit framework.
- A6: Báo cáo thêm root-cause groups, không chỉ finding instances.

## 0. Config

Input:

- ontology artifacts v01
- optional SQLite database path

Output:

- `outputs/audit_findings.json`
- `outputs/test_results.json`
- `outputs/audit_summary.json`
- fixture database nếu cần execution-based semantic tests

In [ ]:
import itertools
import json
import re
import sqlite3
import time
from collections import Counter, defaultdict
from pathlib import Path


workspace_root = Path("/home/baond12/Documents/Research_Cadshouse_Analytics/benchmarks_baond_exp")
ontology_root = workspace_root / "ontology" / "vietnamese_admin_units_object_ontology_v01"

audit_root = workspace_root / "20_q4_ontology_semantic_contract_audit_v01_1"
output_root = audit_root / "outputs"
fixture_root = audit_root / "tests" / "fixtures"

output_root.mkdir(parents=True, exist_ok=True)
fixture_root.mkdir(parents=True, exist_ok=True)

fixture_db_path = fixture_root / "test_admin_units.db"

database_id = "vietnamese_admin_units"

print("workspace_root:", workspace_root)
print("ontology_root:", ontology_root)
print("audit_root:", audit_root)
print("output_root:", output_root)
print("fixture_db_path:", fixture_db_path)
print("ontology_exists:", ontology_root.exists())

## 1. Helpers

Input: common values.

Output: helper functions for loading JSON, saving JSON, adding findings/tests.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def normalize_sql(sql):
    sql = str(sql or "")
    sql = re.sub(r"\s+", " ", sql).strip()
    return sql


def normalize_name(name):
    return str(name or "").strip()


audit_findings = []
test_results = []


def add_finding(category, severity, title, artifact=None, method_id=None, template_id=None, expected=None, actual=None, recommendation=None, status="DETECTED"):
    finding_id = f"AUDIT-{len(audit_findings) + 1:03d}"

    finding = {
        "finding_id": finding_id,
        "category": category,
        "severity": severity,
        "title": title,
        "artifact": artifact,
        "method_id": method_id,
        "template_id": template_id,
        "expected": expected,
        "actual": actual,
        "recommendation": recommendation,
        "status": status,
    }

    audit_findings.append(finding)
    return finding


def add_test_result(test_id, test_name, expected_behavior, actual_behavior, test_status, finding_ids=None, notes=None):
    result = {
        "test_id": test_id,
        "test_name": test_name,
        "expected_behavior": expected_behavior,
        "actual_behavior": actual_behavior,
        "test_status": test_status,
        "finding_ids": finding_ids or [],
        "notes": notes or "",
    }

    test_results.append(result)
    return result

## 2. Load Ontology Artifacts

Input: JSON artifact files.

Output: artifact indexes for audit.

In [ ]:
def load_ontology_artifacts(ontology_root):
    return {
        "classes": load_json(ontology_root / "ontology_classes.json"),
        "properties": load_json(ontology_root / "ontology_properties.json"),
        "relations": load_json(ontology_root / "ontology_relations.json"),
        "methods": load_json(ontology_root / "ontology_methods.json"),
        "intent_rules": load_json(ontology_root / "intent_method_rules.json"),
        "sql_templates": load_json(ontology_root / "sql_templates.json"),
        "unsupported_scope": load_json(ontology_root / "unsupported_scope.json"),
        "runtime_config": load_json(ontology_root / "ontology_runtime_config.json"),
    }


artifacts = load_ontology_artifacts(ontology_root)

class_by_id = {
    item["class_id"]: item
    for item in artifacts["classes"]["classes"]
}

property_by_id = {
    item["property_id"]: item
    for item in artifacts["properties"]["properties"]
}

relation_by_id = {
    item["relation_id"]: item
    for item in artifacts["relations"]["relations"]
}

method_by_id = {
    item["method_id"]: item
    for item in artifacts["methods"]["methods"]
}

sql_template_by_id = {
    item["sql_template_id"]: item
    for item in artifacts["sql_templates"]["sql_templates"]
}

print("classes:", len(class_by_id))
print("properties:", len(property_by_id))
print("relations:", len(relation_by_id))
print("methods:", len(method_by_id))
print("sql_templates:", len(sql_template_by_id))

## 3. Phase 1A - Ontology Consistency Checker

Kiểm tra tham chiếu hợp lệ giữa:

- class -> parent class
- property -> class
- relation -> class/property/inverse relation
- method -> class/relation/sql template
- SQL template -> method
- runtime config -> class/property

In [ ]:
def check_ontology_consistency():
    phase_findings = []

    for class_id, class_item in class_by_id.items():
        parent_class = class_item.get("parent_class")
        if parent_class and parent_class not in class_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Class references missing parent class.",
                artifact="ontology_classes.json",
                expected=f"parent_class {parent_class} exists",
                actual=f"missing parent_class for {class_id}",
                recommendation="Add parent class or fix parent_class reference.",
            ))

    for property_id, property_item in property_by_id.items():
        class_id = property_item.get("class_id")
        if class_id not in class_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Property references missing class.",
                artifact="ontology_properties.json",
                expected=f"class_id {class_id} exists",
                actual=f"missing class for property {property_id}",
                recommendation="Fix class_id or add missing class.",
            ))

    for relation_id, relation_item in relation_by_id.items():
        source_class = relation_item.get("source_class")
        target_class = relation_item.get("target_class")
        source_property = relation_item.get("source_property")
        target_property = relation_item.get("target_property")
        inverse_relation = relation_item.get("inverse_relation")

        if source_class not in class_by_id or target_class not in class_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Relation references missing class.",
                artifact="ontology_relations.json",
                expected="source_class and target_class exist",
                actual={"relation_id": relation_id, "source_class": source_class, "target_class": target_class},
                recommendation="Fix source_class/target_class references.",
            ))

        if source_property not in property_by_id or target_property not in property_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Relation references missing property.",
                artifact="ontology_relations.json",
                expected="source_property and target_property exist",
                actual={"relation_id": relation_id, "source_property": source_property, "target_property": target_property},
                recommendation="Fix relation property references.",
            ))

        if inverse_relation and inverse_relation not in relation_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="HIGH",
                title="Relation references missing inverse relation.",
                artifact="ontology_relations.json",
                expected=f"inverse_relation {inverse_relation} exists",
                actual=f"missing inverse for {relation_id}",
                recommendation="Add inverse relation or remove inverse_relation reference.",
            ))

    for method_id, method_item in method_by_id.items():
        owner_class = method_item.get("owner_class")
        sql_template_id = method_item.get("sql_template_id")

        if owner_class not in class_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Method references missing owner class.",
                artifact="ontology_methods.json",
                method_id=method_id,
                expected=f"owner_class {owner_class} exists",
                actual=f"missing owner_class for {method_id}",
                recommendation="Fix owner_class or add missing class.",
            ))

        for relation_id in method_item.get("required_relations", []):
            if relation_id not in relation_by_id:
                phase_findings.append(add_finding(
                    category="metadata_consistency",
                    severity="CRITICAL",
                    title="Method references missing relation.",
                    artifact="ontology_methods.json",
                    method_id=method_id,
                    expected=f"required_relation {relation_id} exists",
                    actual=f"missing relation for {method_id}",
                    recommendation="Fix required_relations or add missing relation.",
                ))

        if sql_template_id not in sql_template_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="Method references missing SQL template.",
                artifact="ontology_methods.json",
                method_id=method_id,
                template_id=sql_template_id,
                expected=f"sql_template_id {sql_template_id} exists",
                actual="missing SQL template",
                recommendation="Add SQL template or fix method sql_template_id.",
            ))

    for sql_template_id, template_item in sql_template_by_id.items():
        method_id = template_item.get("method_id")
        if method_id not in method_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="CRITICAL",
                title="SQL template references missing method.",
                artifact="sql_templates.json",
                method_id=method_id,
                template_id=sql_template_id,
                expected=f"method_id {method_id} exists",
                actual="missing method for SQL template",
                recommendation="Fix SQL template method_id or add missing method.",
            ))

    default_display_policy = artifacts["runtime_config"].get("default_display_policy", {})
    for class_id, property_name in default_display_policy.items():
        property_id = f"{class_id}.{property_name}"
        if class_id not in class_by_id or property_id not in property_by_id:
            phase_findings.append(add_finding(
                category="metadata_consistency",
                severity="HIGH",
                title="Runtime config references invalid default display property.",
                artifact="ontology_runtime_config.json",
                expected=property_id,
                actual="missing class/property reference",
                recommendation="Fix default_display_policy.",
            ))

    return phase_findings


phase_1a_findings = check_ontology_consistency()
print("Phase 1A findings:", len(phase_1a_findings))

## 4. Phase 1B - Method-Template Contract Checker

Mục tiêu:

- Không để method khai báo parameter nhưng template không có cách preserve.
- Không để optional parameter hợp lệ nhưng không có variant hoặc default policy.
- Kiểm tra projection/filter/template variant coverage ở mức metadata + template string.

Ghi chú:

- Đây là static contract checking, chưa chứng minh SQL đúng về mặt dữ liệu.

In [ ]:
PROPERTY_TO_COLUMN = {
    "name": "name",
    "fullName": "full_name",
    "englishName": "name_en",
    "englishFullName": "full_name_en",
    "codeName": "code_name",
    "englishCodeName": "code_name_en",
}


def method_parameter_catalog(method_item):
    return {
        parameter["name"]: parameter
        for parameter in method_item.get("parameters", [])
    }


def parameter_values_for_contract(parameter):
    allowed_values = parameter.get("allowed_values")
    if allowed_values:
        return allowed_values
    if parameter.get("type") == "integer":
        return [5]
    if parameter.get("type") == "string":
        return ["__sample_value__"]
    return ["__sample_value__"]


def generate_method_parameter_combinations(method_item):
    parameters = method_item.get("parameters", [])
    required_parameters = [item for item in parameters if item.get("required")]
    optional_parameters = [item for item in parameters if not item.get("required")]

    base_items = []
    for parameter in required_parameters:
        values = parameter_values_for_contract(parameter)
        base_items.append([(parameter["name"], value) for value in values])

    if base_items:
        base_combinations = [dict(product) for product in itertools.product(*base_items)]
    else:
        base_combinations = [{}]

    output_combinations = []

    for base in base_combinations:
        output_combinations.append(dict(base))

        for size in range(1, len(optional_parameters) + 1):
            for subset in itertools.combinations(optional_parameters, size):
                value_lists = []

                for parameter in subset:
                    values = parameter_values_for_contract(parameter)
                    value_lists.append([(parameter["name"], value) for value in values])

                for product in itertools.product(*value_lists):
                    params = dict(base)
                    params.update(dict(product))
                    output_combinations.append(params)

    unique = []
    seen = set()

    for combo in output_combinations:
        key = tuple(sorted(combo.items()))
        if key in seen:
            continue
        seen.add(key)
        unique.append(combo)

    return unique


def parameter_key_matches(required_parameter, parameters):
    if "=" in required_parameter:
        name, value = required_parameter.split("=", 1)
        return str(parameters.get(name)) == value
    return required_parameter in parameters


def extract_order_clause(sql):
    match = re.search(r"order\s+by\s+(.+?)(?:\s+limit\s+|;|$)", sql.lower())
    return match.group(1) if match else ""


def extract_limit_value(sql):
    match = re.search(r"\blimit\s+(\d+)\b", sql.lower())
    return int(match.group(1)) if match else None


def sql_select_part(sql):
    sql_norm = normalize_sql(sql).lower()
    return sql_norm.split(" from ", 1)[0]


def sql_contains_exact_identifier(sql_part, identifier):
    pattern = rf"(?<![a-z0-9_]){re.escape(identifier.lower())}(?![a-z0-9_])"
    return re.search(pattern, sql_part.lower()) is not None


def sql_preserves_parameter(sql, parameter_name, parameter_value=None):
    sql_text = str(sql or "")
    sql_norm = normalize_sql(sql_text).lower()

    if parameter_name == "order":
        if "{order}" in sql_norm:
            return True

        order_clause = extract_order_clause(sql_norm)
        expected_order = str(parameter_value or "").lower()

        if expected_order in ["asc", "desc"]:
            return re.search(rf"\b{expected_order}\b", order_clause) is not None

        return bool(order_clause)

    if parameter_name == "limit":
        if "{limit}" in sql_norm:
            return True

        limit_value = extract_limit_value(sql_norm)
        if limit_value is None:
            return False

        return str(limit_value) == str(parameter_value)

    if parameter_name == "outputProperty":
        expected_column = PROPERTY_TO_COLUMN.get(parameter_value)
        if not expected_column:
            return False
        return sql_contains_exact_identifier(sql_select_part(sql_text), expected_column)

    placeholder_patterns = [
        f":{parameter_name}".lower(),
        f"{{{parameter_name}}}".lower(),
    ]

    return any(pattern in sql_norm for pattern in placeholder_patterns)


def variant_preserves_parameters(variant, parameters):
    sql = variant.get("sql", "")
    unfulfilled = []

    # required_parameters decide applicability only.
    # They are not proof that SQL really preserves the semantic parameter.
    for parameter_name, parameter_value in parameters.items():
        if not sql_preserves_parameter(sql, parameter_name, parameter_value):
            unfulfilled.append(parameter_name)

    return unfulfilled


def find_contract_preserving_variant(template_item, parameters):
    candidates = []

    for variant in template_item.get("variants", []):
        required_parameters = variant.get("required_parameters", [])
        required_match = all(parameter_key_matches(item, parameters) for item in required_parameters)

        if not required_match:
            continue

        unfulfilled = variant_preserves_parameters(variant, parameters)
        candidates.append({
            "variant": variant,
            "unfulfilled_parameters": unfulfilled,
        })

    exact_candidates = [
        item for item in candidates
        if not item["unfulfilled_parameters"]
    ]

    if exact_candidates:
        exact_candidates = sorted(
            exact_candidates,
            key=lambda item: len(item["variant"].get("required_parameters", [])),
            reverse=True,
        )
        return exact_candidates[0]

    if candidates:
        candidates = sorted(
            candidates,
            key=lambda item: len(item["unfulfilled_parameters"]),
        )
        return candidates[0]

    return None


def check_method_template_contracts():
    phase_findings = []

    for method_id, method_item in method_by_id.items():
        sql_template_id = method_item.get("sql_template_id")
        template_item = sql_template_by_id.get(sql_template_id)

        if not template_item:
            continue

        combinations = generate_method_parameter_combinations(method_item)

        for parameters in combinations:
            matched = find_contract_preserving_variant(template_item, parameters)

            if matched and not matched["unfulfilled_parameters"]:
                continue

            if matched and matched["unfulfilled_parameters"]:
                phase_findings.append(add_finding(
                    category="method_template_contract",
                    severity="HIGH",
                    title="SQL template variant does not preserve all semantic parameters.",
                    artifact="sql_templates.json",
                    method_id=method_id,
                    template_id=sql_template_id,
                    expected={"parameters_preserved": parameters},
                    actual={
                        "variant_id": matched["variant"].get("variant_id"),
                        "unfulfilled_parameters": matched["unfulfilled_parameters"],
                    },
                    recommendation="Add a SQL variant or template metadata that preserves all semantic parameters.",
                ))
            else:
                phase_findings.append(add_finding(
                    category="template_variant_coverage",
                    severity="HIGH",
                    title="No SQL template variant can satisfy a valid method parameter combination.",
                    artifact="sql_templates.json",
                    method_id=method_id,
                    template_id=sql_template_id,
                    expected={"valid_method_parameters": parameters},
                    actual="no matching SQL variant",
                    recommendation="Add a template variant or declare a default policy for this parameter combination.",
                ))

    return phase_findings


phase_1b_findings = check_method_template_contracts()
print("Phase 1B findings:", len(phase_1b_findings))

## 5. Phase 1C - Runtime Contract Guard

Mục tiêu:

- Method call vi phạm semantic contract phải bị chặn trước execution.
- Không được silently drop semantic parameter.

Output:

- `status = blocked`
- `error_code = SEMANTIC_CONTRACT_VIOLATION`
- `sql_executed = False`

In [ ]:
def validate_parameter_value(parameter_name, parameter_value, parameter_schema):
    parameter_type = parameter_schema.get("type")

    if parameter_value is None:
        return {
            "valid": False,
            "error_code": "INVALID_PARAMETER_VALUE",
            "reason": f"Parameter {parameter_name} must not be null.",
        }

    if parameter_type == "integer":
        if isinstance(parameter_value, bool) or not isinstance(parameter_value, int):
            return {
                "valid": False,
                "error_code": "INVALID_PARAMETER_TYPE",
                "reason": f"Parameter {parameter_name} must be an integer.",
            }

        if parameter_value <= 0:
            return {
                "valid": False,
                "error_code": "INVALID_PARAMETER_RANGE",
                "reason": f"Parameter {parameter_name} must be a positive integer.",
            }

    if parameter_type in ["string", "enum"]:
        if not isinstance(parameter_value, str):
            return {
                "valid": False,
                "error_code": "INVALID_PARAMETER_TYPE",
                "reason": f"Parameter {parameter_name} must be a string.",
            }

        if not parameter_value.strip():
            return {
                "valid": False,
                "error_code": "INVALID_PARAMETER_VALUE",
                "reason": f"Parameter {parameter_name} must not be empty.",
            }

    allowed_values = parameter_schema.get("allowed_values")
    if allowed_values and parameter_value not in allowed_values:
        return {
            "valid": False,
            "error_code": "INVALID_PARAMETER_VALUE",
            "reason": f"Parameter {parameter_name}={parameter_value} is not allowed.",
        }

    return {
        "valid": True,
        "error_code": "",
        "reason": "",
    }


def validate_parameter_dependencies(method_item, parameters):
    # Ranking policy v01.1:
    # - order without limit: valid.
    # - order with limit: valid.
    # - limit without order: invalid for ranked_table methods.
    if method_item.get("return_shape") == "ranked_table":
        if "limit" in parameters and "order" not in parameters:
            return {
                "valid": False,
                "error_code": "PARAMETER_DEPENDENCY_VIOLATION",
                "reason": "Ranking method cannot use limit without order.",
                "unfulfilled_parameters": ["order"],
            }

    return {
        "valid": True,
        "error_code": "",
        "reason": "",
    }


def validate_method_call(method_call):
    method_id = method_call.get("method_id")
    parameters = method_call.get("parameters", {})

    if method_id not in method_by_id:
        return {
            "valid": False,
            "error_code": "UNKNOWN_METHOD",
            "reason": f"Unknown method_id: {method_id}",
        }

    method_item = method_by_id[method_id]
    parameter_catalog = method_parameter_catalog(method_item)

    for parameter_name, parameter_value in parameters.items():
        if parameter_name not in parameter_catalog:
            return {
                "valid": False,
                "error_code": "UNSUPPORTED_PARAMETER",
                "reason": f"Parameter {parameter_name} is not declared by method {method_id}",
                "unfulfilled_parameters": [parameter_name],
            }

        value_validation = validate_parameter_value(
            parameter_name=parameter_name,
            parameter_value=parameter_value,
            parameter_schema=parameter_catalog[parameter_name],
        )

        if not value_validation["valid"]:
            value_validation["unfulfilled_parameters"] = [parameter_name]
            return value_validation

    for parameter_name, parameter in parameter_catalog.items():
        if parameter.get("required") and parameter_name not in parameters:
            return {
                "valid": False,
                "error_code": "MISSING_REQUIRED_PARAMETER",
                "reason": f"Missing required parameter {parameter_name}",
                "unfulfilled_parameters": [parameter_name],
            }

    dependency_validation = validate_parameter_dependencies(method_item, parameters)
    if not dependency_validation["valid"]:
        return dependency_validation

    return {
        "valid": True,
        "error_code": "",
        "reason": "",
    }


def render_guard_sql(sql, parameters):
    rendered_sql = sql

    if "{order}" in rendered_sql:
        rendered_sql = rendered_sql.replace("{order}", str(parameters.get("order", "desc")).upper())

    if "{limit}" in rendered_sql:
        rendered_sql = rendered_sql.replace("{limit}", str(parameters.get("limit", 5)))

    return rendered_sql


def runtime_contract_guard(method_call):
    validation = validate_method_call(method_call)

    if not validation["valid"]:
        return {
            "status": "blocked",
            "error_code": validation["error_code"],
            "reason": validation["reason"],
            "method_call": method_call,
            "unfulfilled_parameters": validation.get("unfulfilled_parameters", []),
            "sql_executed": False,
        }

    method_item = method_by_id[method_call["method_id"]]
    template_item = sql_template_by_id[method_item["sql_template_id"]]
    parameters = method_call.get("parameters", {})
    matched = find_contract_preserving_variant(template_item, parameters)

    if not matched:
        return {
            "status": "blocked",
            "error_code": "SEMANTIC_CONTRACT_VIOLATION",
            "reason": "No SQL variant can preserve all semantic parameters.",
            "method_call": method_call,
            "unfulfilled_parameters": list(parameters.keys()),
            "sql_executed": False,
        }

    if matched["unfulfilled_parameters"]:
        return {
            "status": "blocked",
            "error_code": "SEMANTIC_CONTRACT_VIOLATION",
            "reason": "SQL variant does not preserve all semantic parameters.",
            "method_call": method_call,
            "variant_id": matched["variant"].get("variant_id"),
            "unfulfilled_parameters": matched["unfulfilled_parameters"],
            "sql_executed": False,
        }

    return {
        "status": "allowed",
        "error_code": "",
        "reason": "",
        "method_call": method_call,
        "variant_id": matched["variant"].get("variant_id"),
        "sql": render_guard_sql(matched["variant"].get("sql"), parameters),
        "sql_executed": False,
    }


guard_demo_call = {
    "method_id": "CommuneLevelUnit.list",
    "parameters": {
        "outputProperty": "name",
        "administrativeUnitType": "Phường",
    },
}

guard_demo_result = runtime_contract_guard(guard_demo_call)
guard_demo_result

## 6. Execution Fixture for Behavioral Contract Testing

Một số lỗi không thể chứng minh chỉ bằng metadata.

Ví dụ:

- `include_zero_count = true`
- ranking population là tất cả Province
- SQL dùng INNER JOIN làm mất Province có count = 0

Cell này tạo fixture database nhỏ để kiểm chứng hành vi đó.

In [ ]:
def reset_fixture_database(fixture_db_path):
    if fixture_db_path.exists():
        fixture_db_path.unlink()

    conn = sqlite3.connect(fixture_db_path)
    cur = conn.cursor()

    cur.execute("CREATE TABLE administrative_units (id INTEGER PRIMARY KEY, full_name TEXT)")
    cur.execute("CREATE TABLE provinces (code TEXT PRIMARY KEY, full_name TEXT, administrative_unit_id INTEGER)")
    cur.execute("CREATE TABLE wards (code TEXT PRIMARY KEY, full_name TEXT, province_code TEXT, administrative_unit_id INTEGER)")

    cur.executemany(
        "INSERT INTO administrative_units (id, full_name) VALUES (?, ?)",
        [
            (1, "Tỉnh"),
            (2, "Thành phố trực thuộc trung ương"),
            (3, "Phường"),
            (4, "Xã"),
        ],
    )

    cur.executemany(
        "INSERT INTO provinces (code, full_name, administrative_unit_id) VALUES (?, ?, ?)",
        [
            ("01", "Tỉnh Không Có Phường", 1),
            ("02", "Tỉnh Có Một Phường", 1),
        ],
    )

    cur.executemany(
        "INSERT INTO wards (code, full_name, province_code, administrative_unit_id) VALUES (?, ?, ?, ?)",
        [
            ("00001", "Phường A", "02", 3),
            ("00002", "Xã B", "01", 4),
        ],
    )

    conn.commit()
    conn.close()


def execute_query(db_path, sql, params=None):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    rows = cur.execute(sql, params or {}).fetchall()
    records = [dict(row) for row in rows]
    conn.close()
    return records


reset_fixture_database(fixture_db_path)
print("fixture_db_exists:", fixture_db_path.exists())

## 7. Phase 1D - Contract-driven Test Suite

Test result khác audit finding:

- `test_status = PASS`: test framework phát hiện đúng kỳ vọng.
- `finding_severity = HIGH`: ontology/runtime vẫn có lỗi cần sửa.

In [ ]:
def find_latest_finding(category, method_id=None):
    for finding in reversed(audit_findings):
        if finding["category"] != category:
            continue
        if method_id and finding.get("method_id") != method_id:
            continue
        return finding
    return None


def relation_exists_between(source_class, target_class):
    for relation in relation_by_id.values():
        if relation.get("source_class") == source_class and relation.get("target_class") == target_class:
            return True
    return False


def unsupported_relation_declared(scope_id):
    for relation in artifacts["relations"].get("unsupported_relations", []):
        if scope_id == "missing_region_province_relation":
            relation_id = relation.get("relation_id", "")
            if "AdministrativeRegion" in relation_id or "AdministrativeRegion" in relation.get("reason", ""):
                return True
    return False


def plan_simple_composition(question):
    text = str(question).lower()
    method_calls = []

    if "bao nhiêu tỉnh" in text or "bao nhiêu cấp tỉnh" in text:
        method_calls.append({"method_id": "Province.count", "parameters": {}})

    if "bao nhiêu vùng hành chính" in text or "bao nhiêu vùng địa lý" in text:
        method_calls.append({"method_id": "AdministrativeRegion.count", "parameters": {}})

    return method_calls


def test_mutated_variant_detection():
    mutation_tests = [
        {
            "test_id": "M01",
            "test_name": "mutation_missing_order_by",
            "variant": {
                "variant_id": "mut_missing_order",
                "required_parameters": ["order"],
                "sql": "SELECT p.full_name AS province, COUNT(w.code) AS commune_level_unit_count FROM provinces p JOIN wards w ON w.province_code = p.code GROUP BY p.code, p.full_name;",
            },
            "parameters": {"order": "asc"},
            "expected_unfulfilled": ["order"],
        },
        {
            "test_id": "M02",
            "test_name": "mutation_missing_child_filter",
            "variant": {
                "variant_id": "mut_missing_child_filter",
                "required_parameters": ["childAdministrativeUnitType", "order", "limit"],
                "sql": "SELECT p.full_name AS province, COUNT(w.code) AS commune_level_unit_count FROM provinces p JOIN wards w ON w.province_code = p.code GROUP BY p.code, p.full_name ORDER BY commune_level_unit_count {order} LIMIT {limit};",
            },
            "parameters": {"childAdministrativeUnitType": "Phường", "order": "desc", "limit": 5},
            "expected_unfulfilled": ["childAdministrativeUnitType"],
        },
        {
            "test_id": "M03",
            "test_name": "mutation_fixed_wrong_order",
            "variant": {
                "variant_id": "mut_fixed_wrong_order",
                "required_parameters": ["order"],
                "sql": "SELECT p.full_name AS province FROM provinces p ORDER BY p.full_name DESC;",
            },
            "parameters": {"order": "asc"},
            "expected_unfulfilled": ["order"],
        },
        {
            "test_id": "M04",
            "test_name": "mutation_fixed_wrong_limit",
            "variant": {
                "variant_id": "mut_fixed_wrong_limit",
                "required_parameters": ["limit"],
                "sql": "SELECT p.full_name AS province FROM provinces p LIMIT 5;",
            },
            "parameters": {"limit": 10},
            "expected_unfulfilled": ["limit"],
        },
    ]

    for item in mutation_tests:
        unfulfilled = variant_preserves_parameters(item["variant"], item["parameters"])
        status = "PASS" if sorted(unfulfilled) == sorted(item["expected_unfulfilled"]) else "FAIL"

        add_test_result(
            test_id=item["test_id"],
            test_name=item["test_name"],
            expected_behavior=f"Detect unfulfilled parameters {item['expected_unfulfilled']}",
            actual_behavior={"unfulfilled_parameters": unfulfilled, "variant": item["variant"], "parameters": item["parameters"]},
            test_status=status,
            finding_ids=[],
            notes="Negative mutation test for audit framework.",
        )


def test_invalid_limit_values():
    invalid_values = [-3, "abc", 1.5, None]

    for index, value in enumerate(invalid_values, start=1):
        method_call = {
            "method_id": "Province.rankByCommuneLevelUnitCount",
            "parameters": {
                "order": "desc",
                "limit": value,
            },
        }
        guard_result = runtime_contract_guard(method_call)
        status = "PASS" if guard_result["status"] == "blocked" else "FAIL"

        add_test_result(
            test_id=f"L{index:02d}",
            test_name=f"invalid_limit_value_{value}",
            expected_behavior="Reject invalid limit with structured error.",
            actual_behavior=guard_result,
            test_status=status,
            finding_ids=[],
            notes="Parameter validation hardening test.",
        )


def run_contract_tests():
    # T01: projection mismatch should be blocked.
    method_call = {
        "method_id": "CommuneLevelUnit.list",
        "parameters": {
            "outputProperty": "name",
            "administrativeUnitType": "Phường",
        },
    }
    guard_result = runtime_contract_guard(method_call)
    finding = None

    if guard_result["status"] == "blocked" and "outputProperty" in guard_result.get("unfulfilled_parameters", []):
        finding = add_finding(
            category="projection_mismatch",
            severity="HIGH",
            title="Runtime guard blocked method call because outputProperty was not preserved.",
            artifact="sql_templates.json",
            method_id="CommuneLevelUnit.list",
            template_id="sql.commune_level_unit.list",
            expected="outputProperty=name is preserved by SQL projection",
            actual=guard_result,
            recommendation="Add a SQL variant for CommuneLevelUnit.list(outputProperty=name, administrativeUnitType=...).",
        )
        status = "PASS"
    else:
        status = "FAIL"

    add_test_result(
        test_id="T01",
        test_name="projection_mismatch_commune_level_unit_list_name",
        expected_behavior="Block execution because outputProperty=name is not preserved.",
        actual_behavior=guard_result,
        test_status=status,
        finding_ids=[finding["finding_id"]] if finding else [],
    )

    # T02: child filter should be preserved.
    method_call = {
        "method_id": "Province.rankByCommuneLevelUnitCount",
        "parameters": {
            "childAdministrativeUnitType": "Phường",
            "order": "desc",
            "limit": 5,
        },
    }
    guard_result = runtime_contract_guard(method_call)
    sql = guard_result.get("sql", "")
    has_child_filter = ":childAdministrativeUnitType" in sql and "administrative_units" in sql
    status = "PASS" if guard_result["status"] == "allowed" and has_child_filter else "FAIL"

    if status == "FAIL":
        finding = add_finding(
            category="filter_preservation",
            severity="HIGH",
            title="Child administrative unit type filter is not preserved in ranking SQL.",
            artifact="sql_templates.json",
            method_id="Province.rankByCommuneLevelUnitCount",
            template_id="sql.province.rank_by_commune_level_unit_count",
            expected="SQL preserves childAdministrativeUnitType filter.",
            actual=guard_result,
            recommendation="Use SQL variant with child type filter.",
        )
        finding_ids = [finding["finding_id"]]
    else:
        finding_ids = []

    add_test_result(
        test_id="T02",
        test_name="filter_preservation_child_type_rank",
        expected_behavior="Allowed SQL must preserve childAdministrativeUnitType.",
        actual_behavior=guard_result,
        test_status=status,
        finding_ids=finding_ids,
    )

    # T03: parent + child filter requires modeling beyond current method contract.
    method_call = {
        "method_id": "Province.rankByCommuneLevelUnitCount",
        "parameters": {
            "parentAdministrativeUnitType": "Thành phố trực thuộc trung ương",
            "childAdministrativeUnitType": "Phường",
            "order": "desc",
            "limit": 5,
        },
    }
    guard_result = runtime_contract_guard(method_call)

    if guard_result["status"] == "blocked" and guard_result["error_code"] == "UNSUPPORTED_PARAMETER":
        finding = add_finding(
            category="modeling_gap",
            severity="HIGH",
            title="Method contract cannot express simultaneous parent and child administrative unit type filters.",
            artifact="ontology_methods.json",
            method_id="Province.rankByCommuneLevelUnitCount",
            expected="parentAdministrativeUnitType and childAdministrativeUnitType can be represented separately.",
            actual=guard_result,
            recommendation="Extend structured query intent and method contract with scoped filters.",
        )
        status = "PASS"
    else:
        finding = None
        status = "FAIL"

    add_test_result(
        test_id="T03",
        test_name="parent_child_filter_modeling_gap",
        expected_behavior="Block method call and identify unsupported parentAdministrativeUnitType parameter.",
        actual_behavior=guard_result,
        test_status=status,
        finding_ids=[finding["finding_id"]] if finding else [],
    )

    # T04: zero-count behavior requires execution-based test.
    method_call = {
        "method_id": "Province.rankByCommuneLevelUnitCount",
        "parameters": {
            "childAdministrativeUnitType": "Phường",
            "order": "asc",
            "limit": 5,
        },
    }
    guard_result = runtime_contract_guard(method_call)
    records = []
    fixture_has_zero_count = True
    sql_executed = False

    if guard_result["status"] == "allowed":
        records = execute_query(
            fixture_db_path,
            guard_result["sql"],
            {"childAdministrativeUnitType": "Phường"},
        )
        sql_executed = True

    includes_zero_count_province = any(
        row.get("province") == "Tỉnh Không Có Phường"
        and row.get("commune_level_unit_count") == 0
        for row in records
    )

    zero_count_issue_detected = (
        guard_result["status"] == "allowed"
        and sql_executed
        and fixture_has_zero_count
        and not includes_zero_count_province
    )

    if zero_count_issue_detected:
        finding = add_finding(
            category="aggregation_semantics",
            severity="HIGH",
            title="Ranking SQL drops zero-count province in execution fixture.",
            artifact="sql_templates.json",
            method_id="Province.rankByCommuneLevelUnitCount",
            template_id="sql.province.rank_by_commune_level_unit_count",
            expected="Province with zero matching wards is preserved when ranking ascending by count.",
            actual={"records": records, "guard_result": guard_result, "sql_executed": sql_executed},
            recommendation="Define ranking population and include_zero_count policy; use LEFT JOIN with child filter in ON/subquery/conditional aggregation.",
        )
        status = "PASS"
    else:
        finding = None
        status = "FAIL"

    add_test_result(
        test_id="T04",
        test_name="zero_count_ranking_behavior",
        expected_behavior="Detect that current SQL executes and drops zero-count province.",
        actual_behavior={"records": records, "guard_result": guard_result, "sql_executed": sql_executed},
        test_status=status,
        finding_ids=[finding["finding_id"]] if finding else [],
    )

    # T05: unsupported region-province relation should be based on ontology relations.
    relation_exists = relation_exists_between("Province", "AdministrativeRegion") or relation_exists_between("AdministrativeRegion", "Province")
    unsupported_declared = unsupported_relation_declared("missing_region_province_relation")
    status = "PASS" if not relation_exists and unsupported_declared else "FAIL"

    add_test_result(
        test_id="T05",
        test_name="unsupported_region_province_relation",
        expected_behavior="Reject region-province questions because ontology has no Province-AdministrativeRegion relation.",
        actual_behavior={"relation_exists": relation_exists, "unsupported_declared": unsupported_declared},
        test_status=status,
        finding_ids=[],
    )

    # T06: independent method composition plan should be produced and allowed.
    composition_question = "Có bao nhiêu tỉnh và bao nhiêu vùng hành chính?"
    method_calls = plan_simple_composition(composition_question)
    guard_results = [runtime_contract_guard(item) for item in method_calls]
    status = "PASS" if len(method_calls) == 2 and all(item["status"] == "allowed" for item in guard_results) else "FAIL"

    add_test_result(
        test_id="T06",
        test_name="independent_method_composition_plan",
        expected_behavior="Produce two independent atomic method calls and allow both without requiring relation between Province and AdministrativeRegion.",
        actual_behavior={"question": composition_question, "method_calls": method_calls, "guard_results": guard_results},
        test_status=status,
        finding_ids=[],
    )

    test_mutated_variant_detection()
    test_invalid_limit_values()


run_contract_tests()

print("test_results:", len(test_results))
print("audit_findings:", len(audit_findings))

## 8. Audit Summary

Output:

- summary by severity
- summary by category
- test result summary
- audit execution status
- ontology readiness status

In [ ]:
def build_root_cause_groups(audit_findings):
    groups = defaultdict(list)

    for finding in audit_findings:
        group_key = (
            finding.get("category"),
            finding.get("method_id"),
            finding.get("template_id"),
            finding.get("recommendation"),
        )
        groups[group_key].append(finding["finding_id"])

    output = []

    for index, (group_key, finding_ids) in enumerate(groups.items(), start=1):
        category, method_id, template_id, recommendation = group_key
        output.append({
            "root_cause_id": f"RC-{index:03d}",
            "category": category,
            "method_id": method_id,
            "template_id": template_id,
            "recommendation": recommendation,
            "finding_count": len(finding_ids),
            "finding_ids": finding_ids,
        })

    return output


def build_audit_summary():
    severity_counts = Counter(item["severity"] for item in audit_findings)
    category_counts = Counter(item["category"] for item in audit_findings)
    test_status_counts = Counter(item["test_status"] for item in test_results)
    root_cause_groups = build_root_cause_groups(audit_findings)

    audit_execution_status = "COMPLETED"
    ontology_readiness_status = "READY"

    if severity_counts.get("CRITICAL", 0) > 0 or severity_counts.get("HIGH", 0) > 0:
        ontology_readiness_status = "NOT_READY"

    phase_status = {
        "Phase 1A: Ontology Consistency": "COMPLETED",
        "Phase 1B: Method-Template Contract": "COMPLETED",
        "Phase 1C: Runtime Contract Guard": "COMPLETED",
        "Phase 1D: Contract-driven Tests": "COMPLETED",
        "Phase 1E: Audit Framework Hardening": "COMPLETED",
    }

    return {
        "database_id": database_id,
        "audit_name": "Ontology Semantic Contract Audit v01.1",
        "audit_execution_status": audit_execution_status,
        "ontology_readiness_status": ontology_readiness_status,
        "phase_status": phase_status,
        "finding_count": len(audit_findings),
        "root_cause_group_count": len(root_cause_groups),
        "root_cause_groups": root_cause_groups,
        "finding_count_by_severity": dict(severity_counts),
        "finding_count_by_category": dict(category_counts),
        "test_count": len(test_results),
        "test_count_by_status": dict(test_status_counts),
        "output_files": {
            "audit_findings": str(output_root / "audit_findings.json"),
            "test_results": str(output_root / "test_results.json"),
            "audit_summary": str(output_root / "audit_summary.json"),
            "fixture_db": str(fixture_db_path),
        },
    }


audit_summary = build_audit_summary()
audit_summary

## 9. Save Outputs

Output files:

- `audit_findings.json`
- `test_results.json`
- `audit_summary.json`

In [ ]:
audit_findings_path = output_root / "audit_findings.json"
test_results_path = output_root / "test_results.json"
audit_summary_path = output_root / "audit_summary.json"

save_json(audit_findings, audit_findings_path)
save_json(test_results, test_results_path)
save_json(audit_summary, audit_summary_path)

print("SAVED:", audit_findings_path)
print("SAVED:", test_results_path)
print("SAVED:", audit_summary_path)

## 10. Phase 1 Acceptance Report

Tách rõ:

- Audit execution status: framework đã chạy xong chưa?
- Ontology readiness status: ontology hiện tại đã đủ sẵn sàng cho runtime v02 chưa?

In [ ]:
def print_acceptance_report(audit_summary):
    severity_counts = audit_summary["finding_count_by_severity"]
    test_counts = audit_summary["test_count_by_status"]

    print("ONTOLOGY SEMANTIC CONTRACT AUDIT V01.1")
    print("=" * 60)

    for phase, status in audit_summary["phase_status"].items():
        print(f"{phase}")
        print(f"Status: {status}")
        print()

    print("-" * 60)
    print("Test Cases:", audit_summary["test_count"])
    print("Passed:", test_counts.get("PASS", 0))
    print("Failed:", test_counts.get("FAIL", 0))
    print("Skipped:", test_counts.get("SKIP", 0))
    print()

    print("Audit Findings:")
    print("Critical:", severity_counts.get("CRITICAL", 0))
    print("High:", severity_counts.get("HIGH", 0))
    print("Medium:", severity_counts.get("MEDIUM", 0))
    print("Low:", severity_counts.get("LOW", 0))
    print("Root Cause Groups:", audit_summary["root_cause_group_count"])
    print()

    print("-" * 60)
    print("AUDIT EXECUTION:", audit_summary["audit_execution_status"])
    print("ONTOLOGY READINESS:", audit_summary["ontology_readiness_status"])
    print("=" * 60)


print_acceptance_report(audit_summary)

## 11. Review Tables

Dùng các cell này để review nhanh findings và test results trong notebook.

In [ ]:
for finding in audit_findings:
    print("=" * 100)
    print(finding["finding_id"], finding["severity"], finding["category"])
    print("title:", finding["title"])
    print("method_id:", finding.get("method_id"))
    print("template_id:", finding.get("template_id"))
    print("expected:", finding.get("expected"))
    print("actual:", finding.get("actual"))
    print("recommendation:", finding.get("recommendation"))

In [ ]:
for result in test_results:
    print("=" * 100)
    print(result["test_id"], result["test_status"], result["test_name"])
    print("expected:", result["expected_behavior"])
    print("actual:", result["actual_behavior"])
    print("finding_ids:", result["finding_ids"])